In [ ]:
import turbustat.statistics as tstat
from turbustat.statistics import PowerSpectrum
from astropy.io import fits
from astropy.coordinates import SkyCoord
import numpy as np
import matplotlib.pyplot as plt
#from reproject.mosaicking import find_optimal_celestial_wcs
#from reproject import reproject_interp
#from reproject.mosaicking import reproject_and_coadd
from astropy.wcs import WCS
import astropy.units as u
from astrocut import fits_cut
import gc

## Polarised Intensity (first paper submission)

### Read in data

In [ ]:
# Read in polarised intensity:
hdu_PI_CG = fits.open('/srv/data/cgps-gmims/conv_regrid/PI_CG_conv4_regrd_PI_of_mean.fits')
PI_CG     = hdu_PI_CG[0].data

hdu_PI_G = fits.open('/srv/data/cgps-gmims/conv_regrid/PI_G_regrd_PI_of_mean.fits')
PI_G     = hdu_PI_G[0].data 

hdu_PI_C = fits.open('/srv/data/cgps-gmims/conv_regrid/PI_C_conv4_regrd_PI_of_mean.fits')
PI_C     = hdu_PI_C[0].data 

# Read in RM data:
hdu_RM_CG = fits.open('/srv/data/cgps-gmims/conv_regrid/RM_CG_conv4_regrd.fits')
RM_CG     = hdu_RM_CG[0].data

hdu_RM_G = fits.open('/srv/data/cgps-gmims/conv_regrid/RM_G_regrd.fits')
RM_G     = hdu_RM_G[0].data 

hdu_RM_C = fits.open('/srv/data/cgps-gmims/conv_regrid/RM_C_conv4_regrd.fits')
RM_C     = hdu_RM_C[0].data 

# Mask RM maps for thresholding:
#P_thr   = 0.1 # K
#dRM_thr = 150 # rad/m^2

#RM_C[PI_CG < P_thr] = np.nan
#RM_C[PI_CG < P_thr] = np.nan
#RM_C[PI_CG < P_thr] = np.nan

#RM_C[stderr_CG > dRM_thr] = np.nan



PI_CG[np.isnan(PI_CG)] = 0.0
PI_C[np.isnan(PI_C)]  = 0.0
PI_G[np.isnan(PI_G)]  = 0.0

PI_C[PI_CG == 0.0]  = 0.0
PI_G[PI_CG == 0.0]  = 0.0

RM_CG[np.isnan(RM_CG)] = 0.0
RM_C[np.isnan(RM_C)]  = 0.0
RM_G[np.isnan(RM_G)]  = 0.0

RM_C[PI_CG == 0.0]  = 0.0
RM_G[PI_CG == 0.0]  = 0.0



hdr = hdu_PI_CG[0].header

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(20,5))
ax.imshow(RM_G,vmin=-200,vmax=200,cmap='RdBu_r',origin='lower')

In [ ]:
l = WCS(hdr).all_pix2world(range(PI_CG.shape[1]) ,0, 0)[0]
b = WCS(hdr).all_pix2world(0, range(PI_CG.shape[0]), 0)[1]

print(l)
print(b)

dxy = np.round(hdr['CDELT2'],5)

print(dxy)

In [ ]:
def make_r(dr0=3):

    r  = []
    r1 = []
    r2 = []

    r.append(dr0/2)
    r1.append(0)
    r2.append(dr0)

    a = np.pi*dr0**2
    print(a)

    while(r[-1] < 20):

        r1.append(r2[-1])
        r2.append(np.sqrt(a/np.pi + r1[-1]**2))
        
        r.append((r1[-1]+r2[-1])/2)

        print(r[-1],r2[-1]-r1[-1])

        dr = r2[-1]-r1[-1]
        #print(dr)

    while(r[-1] < 500):

        r1.append(r2[-1])
        r2.append(r1[-1] + dr)

        r.append((r1[-1]+r2[-1])/2)

        print(r[-1],r2[-1]-r1[-1])
        

    return r, r1, r2

In [ ]:
r, r1, r2 = make_r(dr0=3)

print(len(r))

In [ ]:
def make_power_spectrum(PI_G_sub, PI_C_sub, PI_CG_sub, dxy, r1, r2):

    uv_freq_m = np.fft.fftshift(np.fft.fftfreq(PI_G_sub.shape[1]))/dxy
    uv_freq_n = np.fft.fftshift(np.fft.fftfreq(PI_G_sub.shape[0]))/dxy

    uv_m = uv_freq_m*(3e8/(1420e6))*180/np.pi # corrected Oct 2024
    uv_n = uv_freq_n*(3e8/(1420e6))*180/np.pi # corrected Oct 2024
    print(uv_m.shape,uv_n.shape)

    PI_CG_FFT = np.fft.fft2(PI_CG_sub)
    PI_CG_FFT_shift = np.fft.fftshift(PI_CG_FFT)

    PI_G_FFT = np.fft.fft2(PI_G_sub)
    PI_G_FFT_shift = np.fft.fftshift(PI_G_FFT)

    PI_C_FFT = np.fft.fft2(PI_C_sub)
    PI_C_FFT_shift = np.fft.fftshift(PI_C_FFT)

    xuv, yuv = np.meshgrid(uv_m, uv_n)
    ruv = np.sqrt(xuv**2+yuv**2)

    #r = np.arange(dr/2,400+dr/2,dr)
    ps  = np.empty([3,len(r1)])
    dps = np.empty([3,len(r1)])
    
    for j in range(0,len(r)):
        #r1 = r[j]-dr/2
        #r2 = r[j]+dr/2
        idx = np.where((ruv >= r1[j]) & (ruv <= r2[j]))
        #print(len(idx[0]))
        #print(len(idx[1]))
        #print(ruv[idx])
        ps[0,j]  = np.nanmedian(abs(PI_G_FFT_shift[idx]))
        dps[0,j] = np.nanstd(abs(PI_G_FFT_shift[idx]))

        ps[1,j]  = np.nanmedian(abs(PI_C_FFT_shift[idx]))
        dps[1,j] = np.nanstd(abs(PI_C_FFT_shift[idx]))

        ps[2,j]  = np.nanmedian(abs(PI_CG_FFT_shift[idx]))
        dps[2,j] = np.nanstd(abs(PI_CG_FFT_shift[idx]))
    
    return ps, dps

In [ ]:
llim_list = [[72,52],[104,84],[122,102],[140,120],[158,138],[176,156],[194,174]]
name_list = ['52_72','66_86','84_104','102_122','120_140','138_158','156_176','174_194']

idx_bad = np.where((l>179.25) & (l<180.75))
PI_CG[:,idx_bad] = 0.0
RM_CG[:,idx_bad] = 0.0
#PI_G[:,idx_bad] = 0.0

ps_list = []
dps_list = []

ps_RM_list = []
dps_RM_list = []

for i in range(0,7):

    idxl = np.where((l > llim_list[i][1]) & (l < llim_list[i][0]))[0]
    idxb = np.where((b > -3) & (b < 5))[0]

    #PI_G_sub  = padded_image(PI_G[idxb[0]:idxb[-1],idxl[0]:idxl[-1]],20)
    #PI_C_sub  = padded_image(PI_C[idxb[0]:idxb[-1],idxl[0]:idxl[-1]],20)
    #PI_CG_sub = padded_image(PI_CG[idxb[0]:idxb[-1],idxl[0]:idxl[-1]],20)

    PI_G_sub  = PI_G[idxb[0]:idxb[-1],idxl[0]:idxl[-1]]
    PI_C_sub  = PI_C[idxb[0]:idxb[-1],idxl[0]:idxl[-1]]
    PI_CG_sub = PI_CG[idxb[0]:idxb[-1],idxl[0]:idxl[-1]]

    #RM_G_sub  = RM_G[idxb[0]:idxb[-1],idxl[0]:idxl[-1]]
    #RM_C_sub  = RM_C[idxb[0]:idxb[-1],idxl[0]:idxl[-1]]
    #RM_CG_sub = RM_CG[idxb[0]:idxb[-1],idxl[0]:idxl[-1]]

    print('')

    ps, dps = make_power_spectrum(PI_G_sub, PI_C_sub, PI_CG_sub, dxy, r1, r2)
    #ps_RM, dps_RM = make_power_spectrum(RM_G_sub, RM_C_sub, RM_CG_sub, axs, dxy, r1, r2)

    ps_list.append(ps)
    dps_list.append(dps)

    #ps_RM_list.append(ps_RM)
    #dps_RM_list.append(dps_RM)


In [ ]:
fig, axs = plt.subplots(7,1,figsize=(12,22))
plt.subplots_adjust(left=0.1, bottom=0.035, right=0.99, top=0.99, hspace=0.18)

fs = 20

panels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)', '(g)']

for i in range(0,7):
    
    axs[i].plot(r,ps_list[i][0],label='GMIMS-HBN',ms=7,color='C4',marker="s",alpha=1)
    axs[i].plot(r,ps_list[i][1],label='DRAO ST',ms=7,color='C0',marker="o",alpha=1)
    axs[i].plot(r,ps_list[i][2],label='combined data',ms=7,color='C3',marker="d",alpha=1)

    axs[i].plot(np.array(r),(2e4)*np.array(r)**(-1),linestyle='dashed',linewidth=2,color='k')

    #axs[i].fill_between(r, ps_list[i][0]-dps_list[i][0], ps_list[i][0]+dps_list[i][0], color='C4',alpha=0.5)
    #axs[i].plot(r,ps_list[i][0],color='C4')
    #axs[i].plot(r,ps_list[i][1],color='C0')
    #axs[i].plot(r,ps_list[i][2],color='C3')
    
    axs[i].set_yscale('log')
    axs[i].set_xscale('log')
    axs[i].set_xlim(1,500)
    axs[i].set_ylim(0.1,2e5)
    #axs[i].grid()
    axs[i].axvline(x=8.572,color='k',linestyle='dashed',linewidth=1)
    axs[i].axvline(x=17.144,color='k',linestyle='dashed',linewidth=1)
    axs[i].text(1.5,1,str(llim_list[i][1])+r'$^{\circ}$ < $\ell$ < '+str(llim_list[i][0])+r'$^{\circ}$',fontsize=fs+4)
    axs[i].text(1.1,2e4,panels[i],fontsize=fs+4)
    axs[i].tick_params(axis='y', which='minor', left=False,  right=False)
    axs[i].tick_params(axis='y', which='major', left=True,   right=True, width=2, length=6, labelsize=fs)
    axs[i].tick_params(axis='x', which='both',  bottom=True, top=True,   width=2, length=6, labelsize=fs)
    axs[i].set_ylabel('Amplitude',fontsize=fs)
    axs[i].axvspan(100, 200, alpha=0.3, color='grey',zorder=2)
    axs[i].axvspan(200, 500, alpha=0.5, color='grey',zorder=2)
    

    for spine in axs[i].spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2)

axs[6].set_xlabel('Baseline (m)',fontsize=fs)
axs[0].legend(fontsize=fs,markerscale=1,loc='upper right')

plt.savefig('../plots/power_spectra_CGPS_GMIMS_v2.pdf')

In [ ]:
plt.plot(np.array(r),(1e5)*np.array(r)**(-4/3))
plt.xscale('log')
plt.yscale('log')

In [ ]:
fig, axs = plt.subplots(7,1,figsize=(12,20))
plt.subplots_adjust(left=0.08, bottom=0.03, right=0.99, top=0.99, hspace=0.15)

fs = 14

panels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)', '(g)']

for i in range(0,7):
    
    axs[i].plot(r,ps_RM_list[i][0],label='SA',ms=7,color='C4',marker="s",alpha=1)
    axs[i].plot(r,ps_RM_list[i][1],label='AS',ms=7,color='C0',marker="o",alpha=1)
    axs[i].plot(r,ps_RM_list[i][2],label='SA+AS',ms=7,color='C3',marker="d",alpha=1)
in
    #axs[i].fill_between(r, ps_list[i][0]-dps_list[i][0], ps_list[i][0]+dps_list[i][0], color='C4',alpha=0.5)
    #axs[i].plot(r,ps_list[i][0],color='C4')
    #axs[i].plot(r,ps_list[i][1],color='C0')
    #axs[i].plot(r,ps_list[i][2],color='C3')
    
    axs[i].set_yscale('log')
    axs[i].set_xscale('log')
    axs[i].set_xlim(1,500)
    axs[i].set_ylim(1,1e8)
    axs[i].legend(fontsize=fs,markerscale=1)
    #axs[i].grid()
    axs[i].axvline(x=8.572,color='k',linestyle='dashed',linewidth=1)
    axs[i].axvline(x=17.144,color='k',linestyle='dashed',linewidth=1)
    axs[i].text(1.5,1,str(llim_list[i][1])+r'$^{\circ}$ < $\ell$ < '+str(llim_list[i][0])+r'$^{\circ}$',fontsize=fs+4)
    axs[i].text(1.1,3e4,panels[i],fontsize=fs+4)
    axs[i].tick_params(axis='y', which='minor', left=False,  right=False)
    axs[i].tick_params(axis='y', which='major', left=True,   right=True, width=2, length=6, labelsize=fs)
    axs[i].tick_params(axis='x', which='both',  bottom=True, top=True,   width=2, length=6, labelsize=fs)
    axs[i].set_ylabel('Amplitude',fontsize=fs)

    for spine in axs[i].spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2)

axs[6].set_xlabel('Baseline (m)',fontsize=fs)

plt.savefig('../plots/power_spectra_RM_CGPS_GMIMS.pdf')

In [ ]:
fig, axs = plt.subplots(7,1,figsize=(12,20))
plt.subplots_adjust(left=0.08, bottom=0.03, right=0.99, top=0.99, hspace=0.15)

fs = 14

panels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)', '(g)']

for i in range(0,7):
    
    axs[i].plot(r,ps_list[i][2]-ps_list[i][1],label='SA contribution',ms=7,color='C4',marker="s",alpha=1)
    axs[i].plot(r,ps_list[i][2]-ps_list[i][0],label='AS contribution',ms=7,color='C0',marker="o",alpha=1)
    #axs[i].plot(r,ps_list[i][2],label='SA+AS',ms=7,color='C3',marker="d",alpha=1)

    #axs[i].fill_between(r, ps_list[i][0]-dps_list[i][0], ps_list[i][0]+dps_list[i][0], color='C4',alpha=0.5)
    #axs[i].plot(r,ps_list[i][0],color='C4')
    #axs[i].plot(r,ps_list[i][1],color='C0')
    #axs[i].plot(r,ps_list[i][2],color='C3')
    
    #axs[i].set_yscale('log')
    axs[i].set_xscale('log')
    axs[i].set_xlim(1,500)
    #axs[i].set_ylim(0.1,1e5)
    axs[i].set_ylim(0,8000)
    axs[i].legend(fontsize=fs,markerscale=1)
    #axs[i].grid()
    axs[i].axvline(x=8.572,color='k',linestyle='dashed',linewidth=1)
    axs[i].axvline(x=17.144,color='k',linestyle='dashed',linewidth=1)
    #axs[i].text(1.5,1,str(llim_list[i][1])+r'$^{\circ}$ < $\ell$ < '+str(llim_list[i][0])+r'$^{\circ}$',fontsize=fs+4)
    #axs[i].text(1.1,3e4,panels[i],fontsize=fs+4)
    axs[i].tick_params(axis='y', which='minor', left=False,  right=False)
    axs[i].tick_params(axis='y', which='major', left=True,   right=True, width=2, length=6, labelsize=fs)
    axs[i].tick_params(axis='x', which='both',  bottom=True, top=True,   width=2, length=6, labelsize=fs)
    axs[i].set_ylabel('Amplitude',fontsize=fs)

    for spine in axs[i].spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2)

axs[6].set_xlabel('Baseline (m)',fontsize=fs)

plt.savefig('../plots/power_spectra_diffs_CGPS_GMIMS.pdf')

In [ ]:
f = 1420e6
lbd = (2.998e8)/f
print(lbd)
res = (lbd/617.2)*1.22*(180/np.pi)*3600
print(res)

In [ ]:
58/86

In [ ]:
620/(3/(58/60))

In [ ]:
1786/60

In [ ]:
lbd1sq = (3e8/1407e6)**2
lbd2sq = (3e8/1414e6)**2

print(1.9/(lbd1sq-lbd2sq))

## Stokes Q and U (referee's suggestion)

In [ ]:
def read_files_4channels(directory,stokes,filetype):

    band = ['A','B','C','D']
    data_list = []
    hdr_list = []
    print('Reading in '+stokes+' for filetype: '+filetype)

    for i in range(0,4):
        print('band '+band[i])
        hdu = fits.open(directory+stokes+band[i]+'_'+filetype+'.fits')
        data_list.append(hdu[0].data)

        hdr = fits.Header()
        for card in hdu[0].header.cards:
            if card.keyword.strip() != "":
                hdr.append(card)
        hdr['OBJECT'] = stokes+band[i]+'_'+filetype
        hdr_list.append(hdr)
        #print(repr(hdr))
        #print('-------------------')

    gc.collect()

    return data_list,hdr_list

In [ ]:
q_data_list_CG, q_hdr_list_CG = read_files_4channels('/srv/data/cgps-gmims/conv_regrid/','Q','CG_conv4_regrd')
u_data_list_CG, u_hdr_list_CG = read_files_4channels('/srv/data/cgps-gmims/conv_regrid/','U','CG_conv4_regrd')

q_data_list_G, q_hdr_list_G = read_files_4channels('/srv/data/cgps-gmims/conv_regrid/','Q','G_regrd')
u_data_list_G, u_hdr_list_G = read_files_4channels('/srv/data/cgps-gmims/conv_regrid/','U','G_regrd')

q_data_list_C, q_hdr_list_C = read_files_4channels('/srv/data/cgps-gmims/conv_regrid/','Q','C_conv4_regrd')
u_data_list_C, u_hdr_list_C = read_files_4channels('/srv/data/cgps-gmims/conv_regrid/','U','C_conv4_regrd')

q_mean_CG = (q_data_list_CG[0]+q_data_list_CG[1]+q_data_list_CG[2]+q_data_list_CG[3])/4
u_mean_CG = (u_data_list_CG[0]+u_data_list_CG[1]+u_data_list_CG[2]+u_data_list_CG[3])/4

q_mean_G = (q_data_list_G[0]+q_data_list_G[1]+q_data_list_G[2]+q_data_list_G[3])/4
u_mean_G = (u_data_list_G[0]+u_data_list_G[1]+u_data_list_G[2]+u_data_list_G[3])/4

q_mean_C = (q_data_list_C[0]+q_data_list_C[1]+q_data_list_C[2]+q_data_list_C[3])/4
u_mean_C = (u_data_list_C[0]+u_data_list_C[1]+u_data_list_C[2]+u_data_list_C[3])/4


q_mean_CG[np.isnan(q_mean_CG)] = 0.0
q_mean_G[np.isnan(q_mean_G)]  = 0.0
q_mean_C[np.isnan(q_mean_C)]  = 0.0

q_mean_C[q_mean_CG == 0.0]  = 0.0
q_mean_G[q_mean_CG == 0.0]  = 0.0

u_mean_CG[np.isnan(u_mean_CG)] = 0.0
u_mean_G[np.isnan(u_mean_G)]  = 0.0
u_mean_C[np.isnan(u_mean_C)]  = 0.0

u_mean_C[u_mean_CG == 0.0]  = 0.0
u_mean_G[u_mean_CG == 0.0]  = 0.0

In [ ]:
fig,ax = plt.subplots(6,1,figsize=(20,16))
ax[0].imshow(q_mean_C,vmin=-0.2,vmax=0.2,cmap='RdBu_r',origin='lower')
ax[1].imshow(u_mean_C,vmin=-0.2,vmax=0.2,cmap='RdBu_r',origin='lower')
ax[2].imshow(q_mean_CG,vmin=-0.5,vmax=0.5,cmap='RdBu_r',origin='lower')
ax[3].imshow(u_mean_CG,vmin=-0.5,vmax=0.5,cmap='RdBu_r',origin='lower')
ax[4].imshow(q_mean_G,vmin=-0.5,vmax=0.5,cmap='RdBu_r',origin='lower')
ax[5].imshow(u_mean_G,vmin=-0.5,vmax=0.5,cmap='RdBu_r',origin='lower')

In [ ]:
llim_list = [[72,52],[104,84],[122,102],[140,120],[158,138],[176,156],[194,174]]
name_list = ['52_72','66_86','84_104','102_122','120_140','138_158','156_176','174_194']

idx_bad = np.where((l>179.25) & (l<180.75))
q_mean_C[:,idx_bad] = 0.0
u_mean_C[:,idx_bad] = 0.0
q_mean_CG[:,idx_bad] = 0.0
u_mean_CG[:,idx_bad] = 0.0
q_mean_G[:,idx_bad] = 0.0
u_mean_G[:,idx_bad] = 0.0

q_ps_list = []
q_dps_list = []

u_ps_list = []
u_dps_list = []

for i in range(0,7):

    idxl = np.where((l > llim_list[i][1]) & (l < llim_list[i][0]))[0]
    idxb = np.where((b > -3) & (b < 5))[0]

    # Stokes Q
    G_sub  = q_mean_G[idxb[0]:idxb[-1],idxl[0]:idxl[-1]]
    C_sub  = q_mean_C[idxb[0]:idxb[-1],idxl[0]:idxl[-1]]
    CG_sub = q_mean_CG[idxb[0]:idxb[-1],idxl[0]:idxl[-1]]
    print('')
    ps, dps = make_power_spectrum(G_sub, C_sub, CG_sub, dxy, r1, r2)
    q_ps_list.append(ps)
    q_dps_list.append(dps)

    # Stokes U
    G_sub  = u_mean_G[idxb[0]:idxb[-1],idxl[0]:idxl[-1]]
    C_sub  = u_mean_C[idxb[0]:idxb[-1],idxl[0]:idxl[-1]]
    CG_sub = u_mean_CG[idxb[0]:idxb[-1],idxl[0]:idxl[-1]]
    print('')
    ps, dps = make_power_spectrum(G_sub, C_sub, CG_sub, dxy, r1, r2)
    u_ps_list.append(ps)
    u_dps_list.append(dps)
    

In [ ]:
fig, axs = plt.subplots(7,2,figsize=(20,22))
plt.subplots_adjust(left=0.08, bottom=0.035, right=0.99, top=0.97, hspace=0.18, wspace=0.05)

fs = 20

panels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)', '(g)']

for i in range(0,7):

    axs[i,0].plot(r,q_ps_list[i][0],label='GMIMS-HBN',ms=7,color='C4',marker="s",alpha=1)
    axs[i,0].plot(r,q_ps_list[i][1],label='DRAO ST',ms=7,color='C0',marker="o",alpha=1)
    axs[i,0].plot(r,q_ps_list[i][2],label='combined data',ms=7,color='C3',marker="d",alpha=1)
    
    axs[i,1].plot(r,u_ps_list[i][0],label='GMIMS-HBN',ms=7,color='C4',marker="s",alpha=1)
    axs[i,1].plot(r,u_ps_list[i][1],label='DRAO ST',ms=7,color='C0',marker="o",alpha=1)
    axs[i,1].plot(r,u_ps_list[i][2],label='combined data',ms=7,color='C3',marker="d",alpha=1)

    for j in range(0,2):
        axs[i,j].plot(np.array(r),(2e4)*np.array(r)**(-1),linestyle='dashed',linewidth=2,color='k')
    
        axs[i,j].set_yscale('log')
        axs[i,j].set_xscale('log')
        axs[i,j].set_xlim(1,500)
        axs[i,j].set_ylim(0.1,2e5)
        axs[i,j].axvline(x=8.572,color='k',linestyle='dashed',linewidth=1)
        axs[i,j].axvline(x=17.144,color='k',linestyle='dashed',linewidth=1)
        axs[i,j].tick_params(axis='y', which='minor', left=False,  right=False)
        axs[i,j].tick_params(axis='y', which='major', left=True,   right=True, width=2, length=6, labelsize=fs)
        axs[i,j].tick_params(axis='x', which='both',  bottom=True, top=True,   width=2, length=6, labelsize=fs)
        axs[i,j].axvspan(100, 200, alpha=0.3, color='grey',zorder=2)
        axs[i,j].axvspan(200, 500, alpha=0.5, color='grey',zorder=2)
        
        for spine in axs[i,j].spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)
            
    axs[i,0].set_ylabel('Amplitude',fontsize=fs)
    axs[i,0].text(0.4,1e5,panels[i],fontsize=fs+6)
    axs[i,0].text(1.2,1,str(llim_list[i][1])+r'$^{\circ}$ < $\ell$ < '+str(llim_list[i][0])+r'$^{\circ}$',fontsize=fs+4)
    axs[i,1].tick_params(axis='y', labelleft=False)  # Remove y-axis labels from right column

axs[6,0].set_xlabel('Baseline (m)',fontsize=fs)
axs[0,0].legend(fontsize=fs,markerscale=1,loc='upper right')

axs[6,1].set_xlabel('Baseline (m)',fontsize=fs)

axs[0,0].set_title('Stokes Q',fontsize=fs+6)
axs[0,1].set_title('Stokes U',fontsize=fs+6)

plt.savefig('../plots/power_spectra_CGPS_GMIMS_StokesQU.pdf')